# Task 3 - Agentic Workflows: Multi-Agent Financial Research System

Requires a Gemini API key stored as the Colab secret `GEMINI_API_KEY`.
Run cells top to bottom.

In [ ]:
!pip install -q yfinance google-generativeai pydantic duckduckgo-search pandas numpy

In [ ]:
import os
from google.colab import userdata
os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')


In [ ]:
import sys
sys.path.append('src')
sys.path.append('../task1_financial/src')  # single_agent/multi_agent reuse Task 1 indicators

TICKER = 'AAPL'

## Task 3A - Single Autonomous Research Agent

In [ ]:
from single_agent import run_agent

query = (
    f"Analyse the current financial health and market sentiment of {TICKER}. "
    "Identify the top three risks to its share price over the next 90 days "
    "and suggest one data-driven hedge strategy."
)
single_agent_result = run_agent(query, verbose=True)
print('\n\n=== Tool call sequence chosen autonomously by the agent ===')
for step in single_agent_result['trace']:
    print(f"step {step['step']}: {step['tool']}({step['args']})")

## Task 3C (part 1) - Short-Term Memory Demo

Ask a follow-up question that reuses a tool result already in the trace
above, without re-calling the tool.

In [ ]:
from memory import ShortTermMemory
from tools import get_price_data

stm = ShortTermMemory()

# First 'question': what's the RSI?
result1, was_cached1 = stm.get_or_call(get_price_data, TICKER)
print('First call (should NOT be cached):', was_cached1)

# Follow-up 'question': what's the 50-day SMA? -- same underlying tool call,
# answered from the cached result instead of hitting yfinance again.
result2, was_cached2 = stm.get_or_call(get_price_data, TICKER)
print('Follow-up call (SHOULD be cached, no re-fetch):', was_cached2)
print('RSI from cached result:', result2.get('indicators', {}).get('RSI_14'))

## Task 3B - Multi-Agent Coordination with Critique Loop

In [ ]:
from multi_agent import run_multi_agent_pipeline

final_report = run_multi_agent_pipeline(TICKER)
print('\n\n=== FINAL STRUCTURED REPORT ===')
print(final_report.model_dump_json(indent=2))

## Task 3C (part 2) - Persistent Cache Demo

Save the finished brief keyed by ticker + date. A second run today for the
same ticker should load from cache instead of re-running every tool.

In [ ]:
from memory import PersistentCache

cache = PersistentCache()
path = cache.save(TICKER, final_report.model_dump())
print(f'Saved research brief to {path}')

# Simulate a second run on the same day
cached_brief = cache.load(TICKER)
if cached_brief is not None:
    print('Cache HIT -- loaded brief without re-running the pipeline:')
    print(cached_brief)
else:
    print('Cache MISS -- would run the full pipeline.')

## Task 3C (part 3) - Observability Trace

Every tool call across this whole notebook run has been logged to
`logs/agent_trace.jsonl` by the `@observed_tool` decorator.

In [ ]:
import json
with open('logs/agent_trace.jsonl') as f:
    lines = [json.loads(l) for l in f]
print(f'{len(lines)} tool calls logged this session')
for entry in lines[-5:]:
    print(f"{entry['tool']:<22} {entry['duration_ms']:>7} ms  args={entry['args']} {entry['kwargs']}")